# Completeness

According to the HealthBench paper, Maverick's least performant axis was "completeness". What sorts of rubric criteria fall under that axis? What are some the reasons why the judge thinks the responses are not complete?

This notebook exists to sample benchmark results for manual review.

In [4]:
import json
from pathlib import Path

import polars as pl

from simple_evals.improvement.models.results import AllResults
from simple_evals.improvement.paths import RESULTS_DIR

In [5]:
baseline_results_path = RESULTS_DIR / (
    "5df4ba309cb03369f6663786ae6a9904385524a9/"
    + "maverick/healthbench_llama-4-maverick_20251023_212754_allresults.json"
)
baseline_results = AllResults.from_file(baseline_results_path)

In [ ]:
out_dir = Path("sampled")
count = 0


def tags_contains_context_awareness(tags: list[str]) -> bool:
    for tag in tags:
        if tag == "axis:context_awareness":
            return True
    return False


for example in baseline_results.metadata.example_level_metadata:
    # Find examples of "context_awareness" rubric items and write them out along with
    # the prompt_id, the prompt, and the completion.
    rubric_items = []
    for rubric_item in example.rubric_items:
        if tags_contains_context_awareness(rubric_item.tags):
            rubric_items.append(rubric_item)
    if len(rubric_items) == 0:
        continue
    count += 1
    if count <= 20:
        continue
    if count > 30:
        break
    # if count > 10:
    #     break

    info = {
        "prompt_id": example.prompt_id,
        "prompt": [turn.model_dump() for turn in example.prompt],
        "completion": example.completion[0].model_dump(),
        "criteria": [c.model_dump() for c in rubric_items],
    }
    out_path = out_dir / f"{example.prompt_id}.json"
    out_path.write_text(json.dumps(info, indent=2))

In [5]:
example_level = pl.DataFrame(baseline_results.metadata.example_level_metadata)

In [6]:
example_level.shape

(5000, 7)

# Get a dataframe with columns
- prompt_id
- conversation
- completion
- completeness criteria

In [ ]:
out_dir = "sampled/prompt_improvement/completeness"